# Weight Forecasting for Athletes Using MacroFactor Data

> **Problem Statement:** Using nutrition and activity data, can we produce a statistically meaningful forecast of bodyweight to help athletes make weight safely and on schedule?

This project houses a personalized weight forecasting pipeline which evaluates predictive performance across RMSE, MAE, and Directional Accuracy to answer this question.

*Note on evaluation metrics: MAPE is excluded despite being a commonly used forecasting metric. Weight change data can approach zero during maintenance phases, making MAPE's division steps produce undefined or astronomically large results. MAE, RMSE, and directional accuracy are used instead.*

| Layer | Component |
|---|---|
| Data | MacroFactor export (weight, nutrition, expenditure) |
| Baseline | SARIMA |
| Multivariate | SARIMAX |
| ML | XGBoost |
| Final | Ensemble |

The architecture is designed to be personal and adaptable. Users import their own MacroFactor .xlsx file, and the pipeline forecasts their weight trajectory based on their nutrition and activity patterns. The framework is demonstrated below using real MacroFactor data to forecast weight for an upcoming competition deadline.

# Importing libraries
**Summary:** Most dependencies are installed / imported here (Google-related libraries are the exception). It must be run before interacting with the notebook to prevent bugs.

In [1]:
# Non-standard
import importlib.util, subprocess, sys

for lib in ['itables', 'ruptures', 'pmdarima', 'plotly', 'xgboost', 'nbformat']:
    if importlib.util.find_spec(lib) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", lib], check=True)

# Standard
import os
import glob
import numpy as np
import pandas as pd

# Data
from itables import show, init_notebook_mode
init_notebook_mode()

# EDA
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ruptures as rpt

# Models
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import xgboost as xgb
import scipy.stats as stats

# Importing data
**Summary:** Two import environments are supported (Google Colab with a mounted Drive, and local or cloned repository environments). Set `USE_DRIVE = True` for Colab and specify your Drive folder name, or USE_DRIVE = False to load directly from the repository directory. The notebook expects data_weight.csv to be located in the same directory as the notebook itself. Descriptive error messages are shown if the file cannot be found in either environment.

In [2]:
# If running on Colab with Drive, set USE_DRIVE = True and specify your folder name.
# If running locally after cloning, set USE_DRIVE = False.
USE_DRIVE = False  # <-- Set to True if using Google Drive

if USE_DRIVE:
    from google.colab import drive
    if not os.path.isdir('/content/drive'):
        drive.mount('/content/drive')
    folder_name = "macrofactor_weight_forecasting"  # <-- Your Drive folder name
    matches = glob.glob(f'/content/drive/MyDrive/Colab Notebooks/**/{folder_name}/data_weight.xlsx', recursive=True)
    if not matches:
        raise FileNotFoundError(f"\033[91m✖ Could not find data_weight.xlsx inside '{folder_name}' on Drive.\033[0m")
    data_path = matches[0]
else:
    # Local / cloned repo — expects: data_weight.xlsx in same directory
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    data_path = os.path.join(notebook_dir, "data_weight.xlsx")
    if not os.path.exists(data_path):
        raise FileNotFoundError(
            f"\033[91m✖ Could not find data_weight.xlsx at {data_path}\n"
            f"  Ensure the file exists in the same directory as the notebook.\033[0m"
        )

# Load each relevant sheet
df_macros = pd.read_excel(data_path, sheet_name='Calories & Macros')
df_weight_trend = pd.read_excel(data_path, sheet_name='Weight Trend')
df_expenditure = pd.read_excel(data_path, sheet_name='Expenditure')
df_scale_weight = pd.read_excel(data_path, sheet_name='Scale Weight')

# Merge into Single DataFrame
df_raw = df_weight_trend.copy()
df_raw = df_raw.merge(df_macros, on='Date', how='left')
df_raw = df_raw.merge(df_expenditure, on='Date', how='left')
df_raw = df_raw.merge(df_scale_weight[['Date', 'Weight (kg)']], on='Date', how='left', suffixes=('', '_scale'))

# Preparing data
**Summary:** Summary: Our raw MacroFactor data is transformed into a suitable format for time-series forecasting. We establish a DateTime index, handle missing nutrition data (days without logging), forward-fill expenditure estimates, and ensure we have a clean, continuous time series ready for modelling.

In [3]:
show(df_raw)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [4]:
# Setting a DateTime index
df_clean = df_raw.copy()
df_clean['Date'] = pd.to_datetime(df_clean['Date'])
df_clean.set_index('Date', inplace=True)
df_clean = df_clean.sort_index()

In [5]:
# Inspecting missing data
print("\nMissing data per column:")
print(df_clean.isnull().sum().sort_values(ascending=False))


Missing data per column:
Weight (kg)          533
Calories (kcal)      361
Fat (g)              361
Carbs (g)            361
Protein (g)          361
Trend Weight (kg)      0
Expenditure            0
dtype: int64


MacroFactor exports include:
- Weight Trend: Complete daily estimates (requires no processing)
- Expenditure: Complete daily estimates (requires no processing)
- Calories & Macros: Only logged days (requires processing)
- Scale Weight: Sporadic weigh-ins (dropped in favour of Weight Trend)

The only missing values we need to process are for Calories & Macros. We have two options:
1. Drop rows with missing nutrition (reduces days but keeps data quality)
2. Interpolate/forward-fill (keeps all days but assumes nutrition patterns)
   
For initial modelling, we'll use Option 1 (logged days only) to avoid making unfounded assumptions about unlogged nutrition.

In [6]:
# Option 1
df_modelling = df_clean.dropna(subset=['Calories (kcal)', 'Protein (g)', 'Carbs (g)', 'Fat (g)'])

In [7]:
# Select modelling features
modelling_cols = [
    'Trend Weight (kg)',
    'Calories (kcal)',
    'Protein (g)',
    'Carbs (g)',
    'Fat (g)',
    'Expenditure'
]

df_modelling = df_modelling[modelling_cols].copy()

# Final null check
print("\nFinal null count:")
print(df_modelling.isnull().sum())


Final null count:
Trend Weight (kg)    0
Calories (kcal)      0
Protein (g)          0
Carbs (g)            0
Fat (g)              0
Expenditure          0
dtype: int64


In [8]:
show(df_modelling)

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


# EDA: Weight trajectory & training phase detection
**Summary:** Summary: We analyze weight change over the full logging period and use changepoint detection to identify distinct training phases (maintenance, cut, bulk). The ruptures library applies binary segmentation to the weight trend, and an elbow plot determines the optimal number of breakpoints. We then visualize these phases and select the most relevant period for model training based on the use case: forecasting weight during active cut phases for boxing weigh-ins.

In [9]:
# Calculating daily weight change
df_modelling['Weight Change (kg)'] = df_modelling['Trend Weight (kg)'].diff()
df_modelling['Caloric Deficit (kcal)'] = df_modelling['Calories (kcal)'] - df_modelling['Expenditure']

In [10]:
# Changepoint detection setup
signal = df_modelling['Trend Weight (kg)'].dropna().values.reshape(-1, 1)
index = df_modelling['Trend Weight (kg)'].dropna().index

algo = rpt.Binseg(model="l2").fit(signal)

In [11]:
# Elbow plot
residuals = []
k_range = range(1, 8)  # Reduced range since we expect fewer phases than stock eras

for k in k_range:
    bps = algo.predict(n_bkps=k)
    residuals.append(algo.cost.sum_of_costs(bps))

fig = go.Figure(go.Scatter(
    x=list(k_range),
    y=residuals,
    mode='lines+markers',
    line=dict(color='steelblue', width=2),
    marker=dict(size=8),
    hovertemplate='Breakpoints: %{x}<br>SSR: %{y:.2f}<extra></extra>'
))

fig.update_layout(
    title='Elbow Method for Optimal Breakpoints (Training Phase Detection)',
    xaxis_title='Number of Breakpoints',
    xaxis=dict(tickmode='linear', dtick=1),
    yaxis_title='Sum of Squared Residuals',
)

fig.show()

In [12]:
# Breakpoint identification
n_bkps = 3  # <-- Adjust based on elbow plot above

weight_bp = []
breakpoints = algo.predict(n_bkps=n_bkps)

for bp in breakpoints[:-1]:
    weight_bp.append(index[bp])
    print(f"Breakpoint detected at: {index[bp].strftime('%Y-%m-%d')}")

Breakpoint detected at: 2025-09-01
Breakpoint detected at: 2026-01-26
Breakpoint detected at: 2026-02-25


In [32]:
# Visualisations
phase_boundaries = [df_modelling.index[0]] + weight_bp + [df_modelling.index[-1]]
phase_colors = [
    'rgba(100,150,255,0.12)',  # Phase 1
    'rgba(255,150,100,0.12)',  # Phase 2
    'rgba(150,255,150,0.12)',  # Phase 3
    'rgba(255,255,100,0.12)',  # Phase 4
]

fig = go.Figure()

# --- Weight Trend ---
fig.add_trace(go.Scatter(
    x=df_modelling.index,
    y=df_modelling['Trend Weight (kg)'],
    name='Weight Trend',
    line=dict(color='darkblue', width=2.5),
    hovertemplate='Date: %{x}<br>Weight: %{y:.2f} kg<extra></extra>'
))

# --- 7-day Moving Average for Smoothness ---
weight_ma7 = df_modelling['Trend Weight (kg)'].rolling(window=7, center=True).mean()
fig.add_trace(go.Scatter(
    x=df_modelling.index,
    y=weight_ma7,
    name='7-Day MA',
    line=dict(color='steelblue', width=1.5, dash='dash'),
    opacity=0.7,
    hovertemplate='7-Day MA: %{y:.2f} kg<extra></extra>'
))

# --- Phase Shading ---
for i in range(len(phase_boundaries) - 1):
    fig.add_vrect(
        x0=phase_boundaries[i], 
        x1=phase_boundaries[i + 1],
        fillcolor=phase_colors[i % len(phase_colors)],
        layer='below',
        line_width=0,
        annotation_text=f'Phase {i + 1}',
        annotation_position='top left',
        annotation_font_size=11,
        annotation_font_color='black'
    )

# --- Breakpoint Lines ---
for i, bp in enumerate(weight_bp):
    fig.add_vline(
        x=bp.timestamp() * 1000,
        line=dict(color='red', dash='dash', width=2)
    )

fig.update_layout(
    title='Weight Trajectory with Detected Training Phases',
    xaxis_title='Date',
    yaxis_title='Weight (kg)',
    height=550,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

The changepoint algorithm has identified distinct phases in the weight data. 
These correspond to different training regimes:

*Note: Phase labels are demonstration-specific. Users should interpret their 
own breakpoints based on their training history and goals.*

**Phase Analysis:**

In [27]:
# Calculate phase statistics
for i in range(len(phase_boundaries) - 1):
    phase_start = phase_boundaries[i]
    phase_end = phase_boundaries[i + 1]
    phase_data = df_modelling.loc[phase_start:phase_end]
    
    weight_change = phase_data['Trend Weight (kg)'].iloc[-1] - phase_data['Trend Weight (kg)'].iloc[0]
    avg_deficit = phase_data['Caloric Deficit (kcal)'].mean()
    duration_days = len(phase_data)
    
    print("-"*32)
    print(f"Phase {i + 1}: {phase_start.strftime('%Y-%m-%d')} - {phase_end.strftime('%Y-%m-%d')}")
    print(f"- Duration: {duration_days} days")
    print(f"- Weight Change: {weight_change:+.2f} kg")
    print(f"- Avg Daily Deficit: {avg_deficit:+.0f} kcal")
    print(f"- Weekly Rate: {(weight_change / duration_days * 7):.2f} kg/week")
    
    # Classify phase
    if abs(weight_change / duration_days * 7) < 0.2:
        phase_type = "Maintenance"
    elif weight_change < 0:
        phase_type = "Cut"
    else:
        phase_type = "Bulk"
    
    print(f"- Classification: {phase_type}")

--------------------------------
Phase 1: 2024-09-24 - 2025-09-01
- Duration: 76 days
- Weight Change: +0.69 kg
- Avg Daily Deficit: -119 kcal
- Weekly Rate: 0.06 kg/week
- Classification: Maintenance
--------------------------------
Phase 2: 2025-09-01 - 2026-01-26
- Duration: 61 days
- Weight Change: +0.12 kg
- Avg Daily Deficit: -235 kcal
- Weekly Rate: 0.01 kg/week
- Classification: Maintenance
--------------------------------
Phase 3: 2026-01-26 - 2026-02-25
- Duration: 26 days
- Weight Change: -0.63 kg
- Avg Daily Deficit: -156 kcal
- Weekly Rate: -0.17 kg/week
- Classification: Maintenance
--------------------------------
Phase 4: 2026-02-25 - 2026-03-27
- Duration: 30 days
- Weight Change: -0.74 kg
- Avg Daily Deficit: -290 kcal
- Weekly Rate: -0.17 kg/week
- Classification: Maintenance


For the boxing weight forecasting use case, we prioritize training on cut phases where weight is actively decreasing in response to caloric deficit. This ensures the model learns the relationship between nutrition variables and weight loss, which is most relevant for making weight on schedule. Based on the phase analysis above, we select `Phase 3` as our primary training window, starting from `2026-01-26`. This phase shows consistent weight loss driven by sustained caloric deficit, making it the most representative period for modelling weight cuts.

In [17]:
optimal_train_start = weight_bp[1]  # <-- ADJUST based on your phase interpretation
print(f"Selected training start date: {optimal_train_start.strftime('%Y-%m-%d')}")

Selected training start date: 2026-01-26


# EDA: Correlation analysis of nutrition variables
**Summary:** We compute a correlation matrix using daily weight change (first difference) rather than absolute weight to ensure stationarity and produce meaningful correlations. We analyze how nutrition variables (calories, macros, expenditure, deficit) relate to weight change and identify multicollinearity among predictors. Unlike the stock portfolio (36 features), we have only 5-6 variables, so we focus on understanding the feature-target relationships and checking for redundancy without needing dimensionality reduction.

In [35]:
# Prepare variables for correlation analysis
corr_data = df_modelling[[
    'Weight Change (kg)',
    'Calories (kcal)',
    'Protein (g)',
    'Carbs (g)',
    'Fat (g)',
    'Expenditure',
    'Caloric Deficit (kcal)'
]].dropna()

# Also compute lagged weight change to check autocorrelation
corr_data['Weight Change (kg) - Lag 1'] = corr_data['Weight Change (kg)'].shift(1)

corr_data = corr_data.dropna()

In [36]:
# Feature-target correlation
corr_matrix = corr_data.corr()
target_corr = corr_matrix['Weight Change (kg)'].drop(['Weight Change (kg)', 'Weight Change (kg) - Lag 1']).sort_values(ascending=False)

def get_corr_color(val):
    """Color code by correlation strength (accounting for negative correlations)"""
    abs_val = abs(val)
    if abs_val >= 0.3:
        return '#e74c3c' if val < 0 else '#2ecc71'  # Strong negative (red) or positive (green)
    elif abs_val >= 0.15:
        return '#f39c12'  # Moderate (orange)
    else:
        return '#95a5a6'  # Weak (gray)

colors = [get_corr_color(val) for val in target_corr.values]

fig = go.Figure(go.Bar(
    x=target_corr.values,
    y=target_corr.index,
    orientation='h',
    marker=dict(color=colors),
    text=[f'{val:+.3f}' for val in target_corr.values],
    textposition='outside',
    hovertemplate='%{y}: %{x:+.3f}<extra></extra>'
))

# Add reference lines
fig.add_vline(x=0, line=dict(color='black', width=1))
fig.add_vline(x=0.3, line=dict(color='#2ecc71', dash='dash', width=1))
fig.add_vline(x=-0.3, line=dict(color='#e74c3c', dash='dash', width=1))

# Legend
fig.add_trace(go.Bar(x=[None], y=[None], marker=dict(color='#2ecc71'), name='Strong Positive (≥ 0.3)'))
fig.add_trace(go.Bar(x=[None], y=[None], marker=dict(color='#e74c3c'), name='Strong Negative (≤ -0.3)'))
fig.add_trace(go.Bar(x=[None], y=[None], marker=dict(color='#f39c12'), name='Moderate (0.15 – 0.3)'))
fig.add_trace(go.Bar(x=[None], y=[None], marker=dict(color='#95a5a6'), name='Weak (< 0.15)'))

fig.update_layout(
    title='Feature Correlations with Weight Change (Daily)',
    xaxis_title='Correlation Coefficient',
    xaxis=dict(range=[-0.5, 0.5]),
    height=500,
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    bargap=0.2
)

fig.show()

In [37]:
# Full correlation heatmap
fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    colorscale='RdBu_r',  # Red-Blue diverging (red = negative, blue = positive)
    zmid=0,  # Center colorscale at zero
    zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
    textfont=dict(size=9),
    hovertemplate='%{y} × %{x}: %{z:.3f}<extra></extra>'
))

fig.update_layout(
    title='Correlation Matrix: Nutrition Variables & Weight Change',
    height=650,
    width=750,
    xaxis=dict(tickangle=45, tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10)),
)

fig.show()

In [40]:
# Interpretation & multicollinearity assessment
for feature, corr_val in target_corr.items():
    strength = "Strong" if abs(corr_val) >= 0.3 else "Moderate" if abs(corr_val) >= 0.15 else "Weak"
    direction = "negative" if corr_val < 0 else "positive"
    print(f"  • {feature}: {corr_val:+.3f} ({strength} {direction})")

  • Expenditure: +0.259 (Moderate positive)
  • Carbs (g): +0.084 (Weak positive)
  • Calories (kcal): +0.048 (Weak positive)
  • Fat (g): +0.013 (Weak positive)
  • Caloric Deficit (kcal): +0.011 (Weak positive)
  • Protein (g): -0.009 (Weak negative)


1. **All correlations are weak** (|r| < 0.3), suggesting weight change on a day-to-day basis is driven by complex, lagged effects rather than same-day nutrition alone. This is expected (weight loss/gain unfolds over days to weeks, not within 24 hours.

2. **Expenditure shows the strongest correlation (+0.259)** with weight change. This was puzzling at first glance: higher expenditure days correlate with weight *gain*, not loss. Possible explanations:
   - High training days: increased glycogen/water retention for recovery
   - High expenditure: compensatory eating (eating more than the deficit)

3. **Caloric Deficit has near-zero correlation (+0.011)**, which appears to contradict thermodynamics. However, this reflects the lag between deficit and measurable weight change:
   - A 500 kcal deficit today doesn't produce measurable fat loss until 3-7 days later
   - Daily weight fluctuations (water, glycogen, digestion) dwarf same-day fat changes
   - This motivates using **lagged features** in our models (e.g., 3-day or 7-day rolling average deficit)

4. **Macro composition shows minimal correlation** (protein, carbs, fat all |r| < 0.1). This suggests that what you eat matters less than *how much* for day-to-day weight change, though macros may affect longer-term body composition.

5. **Implication for modeling:** The weak same-day correlations suggest we need to engineer lagged and rolling features to capture the true relationship between nutrition and weight change. Models will likely perform better with:
   - 3-day, 7-day rolling averages of deficit
   - Lagged weight change (momentum/trend)
   - Cumulative deficit over recent days

In [44]:
# Check for extreme multicollinearity (|r| > 0.95 between predictors)
print("Multicollinearity Check:")

high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.95:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

if high_corr_pairs:
    for feat1, feat2, corr_val in high_corr_pairs:
        print(f"\n!!! {feat1} × {feat2}: {corr_val:.3f} !!!\n")
    print("Consider removing one variable from highly correlated pairs before modeling.")
else:
    print("No extreme multicollinearity detected.")

Multicollinearity Check:

!!! Calories (kcal) × Caloric Deficit (kcal): 0.990 !!!

Consider removing one variable from highly correlated pairs before modeling.


This is mathematically expected since `Calorie Deficit = Calories - Expenditure`.

When two features are nearly perfectly correlated (r > 0.95), keeping both provides no additional information and can destabilize regression models by inflating coefficient standard errors. We will keep only the Calorie Deficit.

Why?
- Caloric Deficit is the fundamental driver of weight change
- Deficit = Calories - Expenditure captures both intake and output in one variable
- For weight forecasting, we care about energy balance (deficit/surplus), not absolute intake alone

In [ ]:
features_to_drop = ['Calories (kcal)']  # <-- Variables to drop here
df_modeling_cleaned = df_modelling.drop(columns=features_to_drop)

# EDA: Rolling volatility
**Summary:** We compute the 30-day rolling volatility for both Nvidia and the sector market average to understand how the target's risk profile fares across our four identified eras. The key finding is that Nvidia's volatility is consistently exceeding the market average, confirming it amplifies market signals as opposed to mirroring them. This means magnitude prediction is an inherently unreliable approach, regardless of our chosen model, and motivates our decision to use directional accuracy as our primary evaluation metric when modelling.

In [20]:
window = 30

daily_returns = df_stocks_clean.pct_change().dropna()
market_returns = daily_returns.drop(columns='NVIDIA').mean(axis=1)
target_returns = daily_returns['NVIDIA']

market_volatility = market_returns.rolling(window=window).std()
target_volatility = target_returns.rolling(window=window).std()
volatility_spread = target_volatility - market_volatility

era_boundaries = [daily_returns.index[0]] + stock_bp + [daily_returns.index[-1]]
era_colors = ['rgba(0,0,255,0.08)', 'rgba(0,128,0,0.08)',
              'rgba(255,0,0,0.08)', 'rgba(255,200,0,0.08)',
              'rgba(128,0,128,0.08)', 'rgba(255,165,0,0.08)']

In [21]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=market_volatility.index, y=market_volatility,
    name='Sector Market Average',
    line=dict(color='red', width=1.5)
))

fig.add_trace(go.Scatter(
    x=target_volatility.index, y=target_volatility,
    name='Target Equity (NVIDIA)',
    line=dict(color='green', width=1.5)
))

for i in range(len(era_boundaries) - 1):
    fig.add_vrect(
        x0=era_boundaries[i], x1=era_boundaries[i + 1],
        fillcolor=era_colors[i % len(era_colors)],
        layer='below', line_width=0,
        annotation_text=f'Era {i + 1}',
        annotation_position='top left',
        annotation_font_size=10
    )

fig.update_layout(
    title='30-Day Rolling Volatility — Target Equity vs Sector Market Average',
    xaxis_title='Date',
    yaxis_title='Volatility (Std of Daily Returns)',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

In [22]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=volatility_spread.index, y=volatility_spread,
    name='Volatility Spread (Target minus Market)',
    line=dict(color='purple', width=1.5)
))

fig.add_hline(y=0, line=dict(color='black', dash='dash', width=1))

for i in range(len(era_boundaries) - 1):
    fig.add_vrect(
        x0=era_boundaries[i], x1=era_boundaries[i + 1],
        fillcolor=era_colors[i % len(era_colors)],
        layer='below', line_width=0,
        annotation_text=f'Era {i + 1}',
        annotation_position='top left',
        annotation_font_size=10
    )

fig.update_layout(
    title='Volatility Spread — Target Equity Excess Volatility over Market Average',
    xaxis_title='Date',
    yaxis_title='Volatility Difference',
    height=450,
    showlegend=True
)

fig.show()

There is one key finding we can take away from this analysis. The target variable's volatility consistently exceeds market averages across all four identified eras. This confirms that the target doesn't mirror but amplifies market signals. We can safely say that magnitude prediction will be unreliable regardless of the chosen modelling approach. Hence, directional accuracy should be our primary evaluation metric going forward. This is far more practically achievable and meaningful given the nature of our findings.

# Modelling: Daily - SARIMA, SARIMAX
**Summary:** We fit a SARIMA model on an 80/20 train/test split of NVIDIA's daily returns as a univariate baseline. As expected, the model performs poorly. This is intended: any improvement from subsequent models is therefore directly attributable to the inclusion of exogenous variables, which is the core of our problem statement. We then apply dimensionality reduction to reduce our 35 exogenous features to 11 and fit a SARIMAX model using the same parameters as our baseline. SARIMAX meaningfully improves on SARIMA in every metric. Our results confirm our hypothesis, that sector-wide signals can produce a statistically meaningful edge over a naive baseline in forecasting equity returns. However, directional accuracy remains low at `50.98%`, consistent with the Efficient Market Hypothesis. This motivates us to resample our data to a weekly frequency to capture a longer-term landscape.

## SARIMA

In [23]:
# Defining our modelling dataset
df_target_dreturns = df_stocks_clean['NVIDIA'].pct_change().dropna()
df_target_dreturns = df_target_dreturns[df_target_dreturns.index >= '2020-05-26']

# Defining our train/test split (80/20)
split = int(len(df_target_dreturns) * 0.8)
train, test = df_target_dreturns.iloc[:split], df_target_dreturns.iloc[split:]

In [24]:
# We use auto_arima to optimise our parameters
auto_model = auto_arima(
    train,
    seasonal=True,
    m=5,
    stepwise=True,
    information_criterion='aic',
    error_action='ignore',
    suppress_warnings=True
)

p, d, q = auto_model.order
P, D, Q, m = auto_model.seasonal_order
print(f"Optimal parameters: {auto_model.order} | Seasonal: {auto_model.seasonal_order}")

Optimal parameters: (0, 0, 1) | Seasonal: (0, 0, 0, 5)


In [25]:
# We fit and evaluate our model
sarima_fitted = SARIMAX(train, order=(p, d, q), seasonal_order=(P, D, Q, m)).fit(disp=False)
sarima_forecast = sarima_fitted.forecast(steps=len(test))

sarima_mae = mean_absolute_error(test, sarima_forecast)
sarima_rmse = np.sqrt(mean_squared_error(test, sarima_forecast))
sarima_directional = np.mean(np.sign(sarima_forecast) == np.sign(test))*100

print(f"MAE: {sarima_mae:.6f}")
print(f"RMSE: {sarima_rmse:.6f}")
print(f"Directional Accuracy: {sarima_directional:.2f}%")

MAE: 0.014614
RMSE: 0.025128
Directional Accuracy: 31.95%


In [26]:
# Plotting our results
fig = go.Figure()

# 1. Training Data
fig.add_trace(go.Scatter(
    x=train.index,
    y=train,
    mode='lines',
    name='Training Data',
    line=dict(color='blue', width=1)
))

# 2. Actual Returns (Test Data)
fig.add_trace(go.Scatter(
    x=test.index,
    y=test,
    mode='lines',
    name='Actual Returns',
    line=dict(color='orange', width=1)
))

# 3. SARIMA Forecast
fig.add_trace(go.Scatter(
    x=test.index,
    y=sarima_forecast,
    mode='lines',
    name='SARIMA Forecast',
    line=dict(color='red', width=1.5, dash='dash') # 'dash' handles the linestyle='--'
))

# Update layout (Title, Labels, and Legend)
fig.update_layout(
    title='SARIMA Forecast - Target Equity Daily Returns',
    xaxis_title='Date',
    yaxis_title='Daily Return',
    width=1100, # Approximate for 14, 6 ratio
    height=500,
    template='plotly_white', # Clean look
    hovermode='x unified'    # Shows all three values when you hover over a date
)

fig.show()

SARIMA has a directional accuracy of sub-50%. This is exceptionally poor. Given the classical model's univariate nature, this is expected. Daily returns approximate a random walk, with past returns carrying little information about the future. With no exploitable structure, our model defaults to predicting the mean, which for returns is approximately zero. This flat forecast is not a failure but the motivation for everything that follows. Any improvements on our established baseline will be attributable to our exogenous variables, directly addressing our problem statement.

## SARIMAX

In [27]:
# Defining our modelling dataset
df_returns_daily = df_stocks_clean.pct_change().dropna()
df_returns_daily = df_returns_daily[df_returns_daily.index >= '2020-05-26']

df_returns_target = df_returns_daily['NVIDIA']
df_returns_exo = df_returns_daily.drop(columns='NVIDIA')

# Defining our train/test split (80/20)
split = int(len(df_returns_target) * 0.8)
df_returns_target_train, df_returns_target_test = df_returns_target.iloc[:split], df_returns_target.iloc[split:]
df_returns_exo_train, df_returns_exo_test = df_returns_exo.iloc[:split], df_returns_exo.iloc[split:]

In [28]:
# PCA dimensionality reduction
scaler = StandardScaler()
df_returns_exo_train_scaled = scaler.fit_transform(df_returns_exo_train)
df_returns_exo_test_scaled = scaler.transform(df_returns_exo_test)

pca = PCA()
pca.fit(df_returns_exo_train_scaled)

cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

In [29]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, len(cumulative_variance) + 1)),
    y=cumulative_variance,
    mode='lines+markers',
    line=dict(width=2),
    marker=dict(size=6),
    name='Cumulative Variance'
))

for threshold, label in [(0.60, '60%'), (0.70, '70%'), (0.80, '80%'), (0.90, '90%')]:
    fig.add_hline(
        y=threshold,
        line=dict(color='grey', dash='dash', width=1),
        annotation_text=label,
        annotation_position='right'
    )

fig.update_layout(
    title='PCA - Cumulative Explained Variance',
    xaxis_title='Number of Components',
    yaxis_title='Cumulative Explained Variance',
    yaxis=dict(tickformat='.0%'),
    showlegend=False
)

fig.show()

In [30]:
# We fit our model
n_components = np.argmax(cumulative_variance >= 0.80) + 1 # <-- informed by PCA plot above
print(f"Components selected at 80% threshold: {n_components}")

pca_final = PCA(n_components=n_components)
df_returns_exo_train_pca = pca_final.fit_transform(df_returns_exo_train_scaled)
df_returns_exo_test_pca = pca_final.transform(df_returns_exo_test_scaled)

p, d, q = auto_model.order
P, D, Q, m = auto_model.seasonal_order

sarimax_fitted = SARIMAX(
    df_returns_target_train,
    exog=df_returns_exo_train_pca,
    order=(p, d, q),
    seasonal_order=(P, D, Q, m)
).fit(disp=False)

sarimax_forecast = sarimax_fitted.forecast(steps=len(df_returns_target_test), exog=df_returns_exo_test_pca)

sarimax_mae = mean_absolute_error(df_returns_target_test, sarimax_forecast)
sarimax_rmse = np.sqrt(mean_squared_error(df_returns_target_test, sarimax_forecast))
sarimax_directional = np.mean(np.sign(sarimax_forecast) == np.sign(df_returns_target_test))*100

print(f"MAE: {sarimax_mae:.6f}")
print(f"RMSE: {sarimax_rmse:.6f}")
print(f"Directional Accuracy: {sarimax_directional:.2f}%")

Components selected at 80% threshold: 11
MAE: 0.011450
RMSE: 0.018143
Directional Accuracy: 50.98%


In [31]:
# Compare SARIMA and SARIMAX
print(f"\n{'Model':<12} {'MAE':>10} {'RMSE':>10} {'Directional Accuracy':>22}")
print(f"{'-'*56}")
print(f"{'SARIMA':<12} {sarima_mae:>10.6f} {sarima_rmse:>10.6f} {sarima_directional:>21.2f}%")
print(f"{'SARIMAX':<12} {sarimax_mae:>10.6f} {sarimax_rmse:>10.6f} {sarimax_directional:>21.2f}%")


Model               MAE       RMSE   Directional Accuracy
--------------------------------------------------------
SARIMA         0.014614   0.025128                 31.95%
SARIMAX        0.011450   0.018143                 50.98%


In [32]:
# Plotting SARIMAX forecast
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_returns_target_train.index, y=df_returns_target_train,
                         name='Training Data', line=dict(color='blue', width=1)))
fig.add_trace(go.Scatter(x=df_returns_target_test.index, y=df_returns_target_test,
                         name='Actual Returns', line=dict(color='orange', width=1)))
fig.add_trace(go.Scatter(x=df_returns_target_test.index, y=sarimax_forecast,
                         name='SARIMAX Forecast', line=dict(color='red', width=2, dash='dash')))
fig.update_layout(title='SARIMAX - Target Equity Daily Returns',
                  xaxis_title='Date',
                  yaxis_title='Daily Return',
                  template='plotly_white', # Clean look
                  hovermode='x unified')   # Shows all three values when you hover over a date)
fig.show()

Our PCA dimensionality reduction reduced our 35 exogenous features to just 11 at the 80% variance threshold. This is a significant reduction in dimensionality which discards noise whilst retaining the underlying signals.

SARIMAX has meaningfully outperformed SARIMA across every metric (which can be confirmed by the forecast plot). As we kept our ARIMA parameters identical, the meaningful improvements shown by SARIMAX over SARIMA can be confidently attributed to our exogenous variables. That is, they carry genuine predictive information about our target.

However, it still has a directional accuracy of `50.98%`, meaning it's not much more useful than a coin flip. Daily returns have proven extremely difficult for this model to predict. This aligns with the Efficient Market Hypothesis which posits that, at short timescales, all publicly available information is already priced into our data. This leaves very little in the way of signals for our model to exploit. Therefore, we move on to resampling our data to a weekly frequency.

# Modelling: Weekly - SARIMA, SARIMAX, XGBoost
**Summary:** We resample our data to a weekly frequency and refit SARIMA, SARIMAX, and a newly introduced XGBoost model on the same 80/20 train/test split. The frequency change produces a marked improvement in directional accuracy across both multivariate models, validating our hypothesis that sector-wide signals are detectable at a weekly rather than daily timescale.

## SARIMA

In [59]:
# Resample to weekly frequency
df_target_wreturns = df_stocks_clean['NVIDIA'].resample('W').last().pct_change().dropna()
df_target_wreturns = df_target_wreturns[df_target_wreturns.index >= '2020-05-26']

# Train/test split (80/20)
split = int(len(df_target_wreturns) * 0.8)
train_w, test_w = df_target_wreturns.iloc[:split], df_target_wreturns.iloc[split:]

# Fit SARIMA
auto_model_w = auto_arima(
    train_w,
    seasonal=True,
    m=4,
    stepwise=True,
    information_criterion='aic',
    error_action='ignore',
    suppress_warnings=True
)

p_w, d_w, q_w = auto_model_w.order
P_w, D_w, Q_w, m_w = auto_model_w.seasonal_order

sarima_fitted_w = SARIMAX(train_w, order=(p_w, d_w, q_w), seasonal_order=(P_w, D_w, Q_w, m_w)).fit(disp=False)
sarima_forecast_w = sarima_fitted_w.forecast(steps=len(test_w)).values

sarima_mae_w = mean_absolute_error(test_w, sarima_forecast_w)
sarima_rmse_w = np.sqrt(mean_squared_error(test_w, sarima_forecast_w))

# Handle null model — zero forecasts have no directional signal
nonzero_mask = sarima_forecast_w != 0
if nonzero_mask.sum() > 0:
    sarima_directional_w = np.mean(np.sign(sarima_forecast_w[nonzero_mask]) == np.sign(test_w.values[nonzero_mask])) * 100
else:
    sarima_directional_w = None
    print("Note: SARIMA selected null model — directional accuracy undefined")

print(f"Optimal parameters: {auto_model_w.order} | Seasonal: {auto_model_w.seasonal_order}")
print(f"MAE: {sarima_mae_w:.6f} | RMSE: {sarima_rmse_w:.6f} | Directional Accuracy: {f'{sarima_directional_w:.2f}%' if sarima_directional_w is not None else 'N/A (null model)'}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train_w.index, y=train_w, name='Training Data', line=dict(color='blue', width=1)))
fig.add_trace(go.Scatter(x=test_w.index, y=test_w, name='Actual Returns', line=dict(color='orange', width=1)))
fig.add_trace(go.Scatter(x=test_w.index, y=sarima_forecast_w, name='SARIMA Forecast', line=dict(color='red', width=1.5, dash='dash')))
fig.update_layout(title='SARIMA Forecast — Target Equity Weekly Returns',
                  xaxis_title='Date', yaxis_title='Weekly Return',
                  template='plotly_white', hovermode='x unified', height=500)
fig.show()

Note: SARIMA selected null model — directional accuracy undefined
Optimal parameters: (0, 0, 0) | Seasonal: (0, 0, 0, 4)
MAE: 0.046279 | RMSE: 0.061660 | Directional Accuracy: N/A (null model)


As with daily data, SARIMA produces a near-flat forecast at weekly frequency. The baseline is now re-established and we proceed to SARIMAX.

## SARIMAX

In [34]:
# Resample exo variables to weekly frequency
df_returns_exo_w = df_stocks_clean.drop(columns='NVIDIA').resample('W').last().pct_change().dropna()
df_returns_exo_w = df_returns_exo_w[df_returns_exo_w.index >= '2020-05-26']
df_returns_exo_w = df_returns_exo_w.loc[df_target_wreturns.index]

# Train/test split (80/20)
df_returns_exo_train_w = df_returns_exo_w.iloc[:split]
df_returns_exo_test_w = df_returns_exo_w.iloc[split:]
df_returns_target_train_w = train_w
df_returns_target_test_w = test_w

# Scaling and PCA
scaler_w = StandardScaler()
df_returns_exo_train_scaled_w = scaler_w.fit_transform(df_returns_exo_train_w)
df_returns_exo_test_scaled_w = scaler_w.transform(df_returns_exo_test_w)

pca_w = PCA()
pca_w.fit(df_returns_exo_train_scaled_w)
cumulative_variance_w = np.cumsum(pca_w.explained_variance_ratio_)

n_components_w = np.argmax(cumulative_variance_w >= 0.80) + 1
print(f"Weekly PCA components at 80% threshold: {n_components_w}")

pca_final_w = PCA(n_components=n_components_w)
df_returns_exo_train_pca_w = pca_final_w.fit_transform(df_returns_exo_train_scaled_w)
df_returns_exo_test_pca_w = pca_final_w.transform(df_returns_exo_test_scaled_w)

Weekly PCA components at 80% threshold: 11


In [35]:
# Fit SARIMAX
auto_model_w = auto_arima(
    df_returns_target_train_w,
    seasonal=True,
    m=4,
    stepwise=True,
    information_criterion='aic',
    error_action='ignore',
    suppress_warnings=True
)

p_w, d_w, q_w = auto_model_w.order
P_w, D_w, Q_w, m_w = auto_model_w.seasonal_order
print(f"Optimal parameters: {auto_model_w.order} | Seasonal: {auto_model_w.seasonal_order}")

sarimax_fitted_w = SARIMAX(
    df_returns_target_train_w,
    exog=df_returns_exo_train_pca_w,
    order=(p_w, d_w, q_w),
    seasonal_order=(P_w, D_w, Q_w, m_w)
).fit(disp=False)

sarimax_forecast_w = sarimax_fitted_w.forecast(
    steps=len(df_returns_target_test_w),
    exog=df_returns_exo_test_pca_w
)

sarimax_mae_w = mean_absolute_error(df_returns_target_test_w, sarimax_forecast_w)
sarimax_rmse_w = np.sqrt(mean_squared_error(df_returns_target_test_w, sarimax_forecast_w))
sarimax_directional_w = np.mean(np.sign(sarimax_forecast_w) == np.sign(df_returns_target_test_w)) * 100

Optimal parameters: (0, 0, 0) | Seasonal: (0, 0, 0, 4)


## XGBoost

In [36]:
# Build lag features
df_pca_train_w = pd.DataFrame(
    df_returns_exo_train_pca_w,
    index=df_returns_target_train_w.index,
    columns=[f'PC{i+1}' for i in range(n_components_w)]
)
df_pca_test_w = pd.DataFrame(
    df_returns_exo_test_pca_w,
    index=df_returns_target_test_w.index,
    columns=[f'PC{i+1}' for i in range(n_components_w)]
)

df_combined_w = pd.concat([
    df_pca_train_w.assign(TARGET=df_returns_target_train_w.values),
    df_pca_test_w.assign(TARGET=df_returns_target_test_w.values)
])

lag_days_pca_w = [1, 2, 3, 4]
lag_days_target_w = [1, 2]

for lag in lag_days_pca_w:
    for col in [f'PC{i+1}' for i in range(n_components_w)]:
        df_combined_w[f'{col}_lag{lag}'] = df_combined_w[col].shift(lag)

for lag in lag_days_target_w:
    df_combined_w[f'TARGET_lag{lag}'] = df_combined_w['TARGET'].shift(lag)

df_combined_w['month'] = df_combined_w.index.month
df_combined_w['quarter'] = df_combined_w.index.quarter
df_combined_w = df_combined_w.drop(columns=[f'PC{i+1}' for i in range(n_components_w)])
df_combined_w = df_combined_w.dropna()

feature_cols_w = [col for col in df_combined_w.columns if col != 'TARGET']
X_w = df_combined_w[feature_cols_w]
y_w = df_combined_w['TARGET']

X_train_w = X_w[X_w.index <= df_returns_target_train_w.index[-1]]
X_test_w = X_w[X_w.index > df_returns_target_train_w.index[-1]]
y_train_w = y_w[y_w.index <= df_returns_target_train_w.index[-1]]
y_test_w = y_w[y_w.index > df_returns_target_train_w.index[-1]]

print(f"XGBoost weekly features: {X_train_w.shape[1]}")
print(f"XGBoost weekly training observations: {len(X_train_w)}")

XGBoost weekly features: 48
XGBoost weekly training observations: 230


In [37]:
# Grid search
param_grid = {
    'max_depth': [2, 3, 4],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

xgb_grid_w = GridSearchCV(
    xgb.XGBRegressor(random_state=42, eval_metric='rmse'),
    param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=0
)

xgb_grid_w.fit(X_train_w, y_train_w)
best_xgb_w = xgb_grid_w.best_estimator_
print(f"Best parameters: {xgb_grid_w.best_params_}")

xgb_forecast_w = best_xgb_w.predict(X_test_w)

xgb_mae_w = mean_absolute_error(y_test_w, xgb_forecast_w)
xgb_rmse_w = np.sqrt(mean_squared_error(y_test_w, xgb_forecast_w))
xgb_directional_w = np.mean(np.sign(xgb_forecast_w) == np.sign(y_test_w)) * 100

Best parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 100, 'subsample': 0.8}


## Comparison, plots, and feature importance

In [38]:
# Comparison
print(f"\n{'Model':<12} {'MAE':>10} {'RMSE':>10} {'Directional Accuracy':>22}")
print(f"{'-'*57}")
print(f"{'SARIMAX':<12} {sarimax_mae_w:>10.6f} {sarimax_rmse_w:>10.6f} {sarimax_directional_w:>21.2f}%")
print(f"{'XGBoost':<12} {xgb_mae_w:>10.6f} {xgb_rmse_w:>10.6f} {xgb_directional_w:>21.2f}%")
print(f"\nNote: Weekly SARIMA excluded from directional comparison — auto_arima selected a null model predicting zero at every step.")

# Plots
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_returns_target_train_w.index, y=df_returns_target_train_w,
                         name='Training Data', line=dict(color='blue', width=1)))
fig.add_trace(go.Scatter(x=df_returns_target_test_w.index, y=df_returns_target_test_w,
                         name='Actual Returns', line=dict(color='orange', width=1.5)))
fig.add_trace(go.Scatter(x=df_returns_target_test_w.index, y=sarimax_forecast_w,
                         name='SARIMAX Forecast', line=dict(color='red', width=2, dash='dash')))
fig.update_layout(title='SARIMAX Weekly Forecast — Target Equity',
                  xaxis_title='Date', yaxis_title='Weekly Return',
                  template='plotly_white', hovermode='x unified', height=500)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_returns_target_train_w.index, y=df_returns_target_train_w,
                         name='Training Data', line=dict(color='blue', width=1)))
fig.add_trace(go.Scatter(x=y_test_w.index, y=y_test_w,
                         name='Actual Returns', line=dict(color='orange', width=1.5)))
fig.add_trace(go.Scatter(x=y_test_w.index, y=xgb_forecast_w,
                         name='XGBoost Forecast', line=dict(color='red', width=2, dash='dash')))
fig.update_layout(title='XGBoost Weekly Forecast — Target Equity',
                  xaxis_title='Date', yaxis_title='Weekly Return',
                  template='plotly_white', hovermode='x unified', height=500)
fig.show()




Model               MAE       RMSE   Directional Accuracy
---------------------------------------------------------
SARIMAX        0.036654   0.047075                 71.19%
XGBoost        0.045266   0.059966                 61.02%

Note: Weekly SARIMA excluded from directional comparison — auto_arima selected a null model predicting zero at every step.


In [39]:
# Feature importance
df_importance_w = pd.DataFrame({
    'Feature': feature_cols_w,
    'Importance': best_xgb_w.feature_importances_
}).sort_values('Importance', ascending=False)

fig = go.Figure(go.Bar(
    x=df_importance_w['Importance'].head(20),
    y=df_importance_w['Feature'].head(20),
    orientation='h',
    marker=dict(color='steelblue')
))
fig.update_layout(
    title='XGBoost Weekly Feature Importance — Top 20',
    xaxis_title='Importance (Gain)',
    yaxis=dict(autorange='reversed'),
    height=600,
    template='plotly_white'
)
fig.show()

Resampling our data from daily to weekly frequencies has proved very effective. Our SARIMAX forecast's directional accuracy has improved from `50.98%` to `71.19%`. This confirms that market signals were present in our exogenous variables, they merely needed time to manifest into our target variable's pricing to be detected.

The additional XGBoost model trails behind the SARIMAX with a directional accuracy of `61.02%`. This model was introduced to see whether or not XGBoost's non-linear capabilities produces superior results when compared with SARIMAX's linear approach. As can be seen, the added complexity was not rewarded, suggesting that there are limited non-linear signals to exploit in the data. Despite this, it remains a valuable ensemble component; its errors are partially decorrelated from SARIMAX's, meaning their combination reduces forecast variance even without introducing genuinely new signal.

With SARIMAX as the strongest individual model, we proceed to ensemble methods to investigate whether combining SARIMAX and XGBoost predictions can push directional accuracy further.

# Modelling: Weekly - Ensemble
**Summary:** We evaluate three initial ensemble approaches (simple average, weighted average, and stacking). The results were all inferior to our SARIMAX model due to correlated errors between them (error decorrelation analysis reveals a theoretical ceiling of 91.5% and motivates a confidence-based switching rule: use SARIMAX when its forecast magnitude exceeds an OOF-derived threshold, otherwise defer to XGBoost). Therefore, we introduce a novel confidence switching method to investigate whether combining SARIMAX and XGBoost predictions outperforms either model individually. This approach achieves `74.58%` directional accuracy, the best result across all models.

In [41]:
# Align SARIMAX and XGBoost forecasts to common test index
common_test_index = df_returns_target_test_w.index.intersection(y_test_w.index)
sarimax_forecast_aligned = pd.Series(sarimax_forecast_w.values, index=df_returns_target_test_w.index)[common_test_index]
xgb_forecast_aligned = pd.Series(xgb_forecast_w, index=y_test_w.index)[common_test_index]
actual_aligned = df_returns_target_test_w[common_test_index]

print(f"Aligned test observations: {len(actual_aligned)}")
print(f"Test period: {actual_aligned.index[0].date()} to {actual_aligned.index[-1].date()}")

Aligned test observations: 59
Test period: 2024-11-24 to 2026-01-04


In [42]:
# Simple average
ensemble_simple = (sarimax_forecast_aligned + xgb_forecast_aligned) / 2
ensemble_simple_mae = mean_absolute_error(actual_aligned, ensemble_simple)
ensemble_simple_rmse = np.sqrt(mean_squared_error(actual_aligned, ensemble_simple))
ensemble_simple_directional = np.mean(np.sign(ensemble_simple) == np.sign(actual_aligned)) * 100

In [43]:
# Weighted average
sarimax_normalised = sarimax_directional_w / (sarimax_directional_w + xgb_directional_w)
xgb_normalised = xgb_directional_w / (sarimax_directional_w + xgb_directional_w)
ensemble_weighted = (sarimax_forecast_aligned * sarimax_normalised) + (xgb_forecast_aligned * xgb_normalised)
ensemble_weighted_mae = mean_absolute_error(actual_aligned, ensemble_weighted)
ensemble_weighted_rmse = np.sqrt(mean_squared_error(actual_aligned, ensemble_weighted))
ensemble_weighted_directional = np.mean(np.sign(ensemble_weighted) == np.sign(actual_aligned)) * 100

In [44]:
# Stacking
from sklearn.linear_model import RidgeCV

n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)
sarimax_oof = np.zeros(len(df_returns_target_train_w))
xgb_oof = np.zeros(len(df_returns_target_train_w))

for fold, (train_idx, val_idx) in enumerate(tscv.split(df_returns_target_train_w)):

    fold_target_train = df_returns_target_train_w.iloc[train_idx]
    fold_target_val = df_returns_target_train_w.iloc[val_idx]
    fold_exo_train = df_returns_exo_train_pca_w[train_idx]
    fold_exo_val = df_returns_exo_train_pca_w[val_idx]

    try:
        fold_sarimax = SARIMAX(
            fold_target_train, exog=fold_exo_train,
            order=(p_w, d_w, q_w), seasonal_order=(P_w, D_w, Q_w, m_w)
        ).fit(disp=False)
        sarimax_oof[val_idx] = fold_sarimax.forecast(steps=len(val_idx), exog=fold_exo_val)
    except:
        sarimax_oof[val_idx] = 0

    fold_pca_train = pd.DataFrame(fold_exo_train, index=fold_target_train.index,
                                   columns=[f'PC{i+1}' for i in range(n_components_w)])
    fold_pca_val = pd.DataFrame(fold_exo_val, index=fold_target_val.index,
                                 columns=[f'PC{i+1}' for i in range(n_components_w)])
    fold_combined = pd.concat([
        fold_pca_train.assign(TARGET=fold_target_train.values),
        fold_pca_val.assign(TARGET=fold_target_val.values)
    ])
    for lag in lag_days_pca_w:
        for col in [f'PC{i+1}' for i in range(n_components_w)]:
            fold_combined[f'{col}_lag{lag}'] = fold_combined[col].shift(lag)
    for lag in lag_days_target_w:
        fold_combined[f'TARGET_lag{lag}'] = fold_combined['TARGET'].shift(lag)
    fold_combined['month'] = fold_combined.index.month
    fold_combined['quarter'] = fold_combined.index.quarter
    fold_combined = fold_combined.drop(columns=[f'PC{i+1}' for i in range(n_components_w)]).dropna()
    fold_feature_cols = [col for col in fold_combined.columns if col != 'TARGET']
    fold_X_train = fold_combined[fold_feature_cols][fold_combined.index <= fold_target_train.index[-1]]
    fold_X_val = fold_combined[fold_feature_cols][fold_combined.index > fold_target_train.index[-1]]
    fold_y_train = fold_combined['TARGET'][fold_combined.index <= fold_target_train.index[-1]]
    if len(fold_X_train) > 10 and len(fold_X_val) > 0:
        fold_xgb = xgb.XGBRegressor(**xgb_grid_w.best_params_, random_state=42, eval_metric='rmse')
        fold_xgb.fit(fold_X_train, fold_y_train, verbose=False)
        xgb_oof[val_idx[:len(fold_X_val)]] = fold_xgb.predict(fold_X_val)

meta_model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
meta_model.fit(np.column_stack([sarimax_oof, xgb_oof]), df_returns_target_train_w.values)
ensemble_stacked = meta_model.predict(np.column_stack([sarimax_forecast_aligned.values, xgb_forecast_aligned.values]))
ensemble_stacked_mae = mean_absolute_error(actual_aligned, ensemble_stacked)
ensemble_stacked_rmse = np.sqrt(mean_squared_error(actual_aligned, ensemble_stacked))
ensemble_stacked_directional = np.mean(np.sign(ensemble_stacked) == np.sign(actual_aligned)) * 100

In [45]:
# Error decorrelation analysis
sarimax_correct = np.sign(sarimax_forecast_aligned) == np.sign(actual_aligned)
xgb_correct = np.sign(xgb_forecast_aligned) == np.sign(actual_aligned)
disagree_mask = np.sign(sarimax_forecast_aligned) != np.sign(xgb_forecast_aligned)

print(f"Both correct: {(sarimax_correct & xgb_correct).sum()} | SARIMAX only: {(sarimax_correct & ~xgb_correct).sum()} | XGBoost only: {(~sarimax_correct & xgb_correct).sum()} | Both wrong: {(~sarimax_correct & ~xgb_correct).sum()}")
print(f"Theoretical ceiling: {(sarimax_correct | xgb_correct).sum()} / {len(actual_aligned)} = {(sarimax_correct | xgb_correct).mean()*100:.2f}%")
print(f"Disagreement weeks: {disagree_mask.sum()} | SARIMAX wins: {(np.sign(sarimax_forecast_aligned[disagree_mask]) == np.sign(actual_aligned[disagree_mask])).sum()} | XGBoost wins: {(np.sign(xgb_forecast_aligned[disagree_mask]) == np.sign(actual_aligned[disagree_mask])).sum()}")

Both correct: 24 | SARIMAX only: 18 | XGBoost only: 12 | Both wrong: 5
Theoretical ceiling: 54 / 59 = 91.53%
Disagreement weeks: 30 | SARIMAX wins: 18 | XGBoost wins: 12


In [46]:
# Confidence switching
sarimax_oof_series = pd.Series(sarimax_oof, index=df_returns_target_train_w.index)
threshold_oof = sarimax_oof_series.abs().mean()
print(f"OOF-derived threshold: {threshold_oof:.4f}")

ensemble_switching = pd.Series(index=actual_aligned.index, dtype=float)
for idx in actual_aligned.index:
    if abs(sarimax_forecast_aligned[idx]) >= threshold_oof:
        ensemble_switching[idx] = sarimax_forecast_aligned[idx]
    else:
        ensemble_switching[idx] = xgb_forecast_aligned[idx]

switching_mae = mean_absolute_error(actual_aligned, ensemble_switching)
switching_rmse = np.sqrt(mean_squared_error(actual_aligned, ensemble_switching))
switching_directional = np.mean(np.sign(ensemble_switching) == np.sign(actual_aligned)) * 100

OOF-derived threshold: 0.0363


In [47]:
# Comparisons

print(f"\n{'Model':<25} {'MAE':>10} {'RMSE':>10} {'Directional Accuracy':>22}")
print(f"{'-'*69}")
print(f"{'SARIMAX':<25} {sarimax_mae_w:>10.6f} {sarimax_rmse_w:>10.6f} {sarimax_directional_w:>21.2f}%")
print(f"{'XGBoost':<25} {xgb_mae_w:>10.6f} {xgb_rmse_w:>10.6f} {xgb_directional_w:>21.2f}%")
print(f"{'Ensemble Simple':<25} {ensemble_simple_mae:>10.6f} {ensemble_simple_rmse:>10.6f} {ensemble_simple_directional:>21.2f}%")
print(f"{'Ensemble Weighted':<25} {ensemble_weighted_mae:>10.6f} {ensemble_weighted_rmse:>10.6f} {ensemble_weighted_directional:>21.2f}%")
print(f"{'Ensemble Stacking':<25} {ensemble_stacked_mae:>10.6f} {ensemble_stacked_rmse:>10.6f} {ensemble_stacked_directional:>21.2f}%")
print(f"{'Confidence Switching':<25} {switching_mae:>10.6f} {switching_rmse:>10.6f} {switching_directional:>21.2f}%")


Model                            MAE       RMSE   Directional Accuracy
---------------------------------------------------------------------
SARIMAX                     0.036654   0.047075                 71.19%
XGBoost                     0.045266   0.059966                 61.02%
Ensemble Simple             0.034959   0.045516                 69.49%
Ensemble Weighted           0.034456   0.044984                 69.49%
Ensemble Stacking           0.035781   0.045952                 71.19%
Confidence Switching        0.034113   0.044272                 74.58%


In [48]:
# Plots

fig = go.Figure()
fig.add_trace(go.Scatter(x=actual_aligned.index, y=actual_aligned,
                         name='Actual Returns', line=dict(color='orange', width=1.5)))
fig.add_trace(go.Scatter(x=actual_aligned.index, y=ensemble_switching,
                         name='Confidence Switching', line=dict(color='red', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=actual_aligned.index, y=sarimax_forecast_aligned,
                         name='SARIMAX', line=dict(color='blue', width=1, dash='dot'), opacity=0.6))
fig.add_trace(go.Scatter(x=actual_aligned.index, y=xgb_forecast_aligned,
                         name='XGBoost', line=dict(color='green', width=1, dash='dot'), opacity=0.6))
fig.add_hline(y=0, line=dict(color='black', dash='dash', width=0.8))
fig.update_layout(title='Confidence Switching Ensemble — Target Equity Weekly Returns',
                  xaxis_title='Date', yaxis_title='Weekly Return',
                  template='plotly_white', hovermode='x unified', height=450)
fig.show()

Standard ensemble methods all failed to improve on SARIMAX's directional accuracy of `73.50%`. This is explained by our error decorrelation analysis in which both models agreed on direction in 29 cases and disagreed in 30 (of 59). When they disagreed, SARIMAX was correct 18 times and XGBoost 12 times. The models are failing on overlapping weeks, meaning averaging their predictions preserves both models' mistakes rather than cancelling them out.

This motivated a more targeted approach. We built a switching rule that selects between models based on SARIMAX's forecast confidence. When SARIMAX predicts confidently (above a threshold derived from out-of-fold predictions on training data), it is trusted. When its forecast is near zero — indicating low confidence — XGBoost is used instead. The threshold of 0.0372 was derived entirely from training data using out-of-fold SARIMAX predictions, ensuring no look-ahead.

The result is `74.58%` directional accuracy, the strongest result across all approaches and metrics (magnitude and direction).

# Model validation
**Summary:** The confidence switching ensemble is confirmed to be statistically significant at p < 0.001. Walk-forward validation across 114 weekly steps, has our directional accuracy holding at `73.50%` (from `74.58%`) which confirms that the result is robust. Residual analysis confirms approximate normality, no significant autocorrelation, and negligible systematic bias. The primary limitation identified is heteroscedastic residual variance, consistent with the regime-dependent volatility identified in our EDA.

## Statistical significance

In [52]:
n_correct = int(switching_directional / 100 * len(actual_aligned))
n_total = len(actual_aligned)
p_value = stats.binomtest(n_correct, n_total, p=0.5, alternative='greater').pvalue

print(f"Model: Confidence Switching Ensemble")
print(f"Correct predictions: {n_correct} / {n_total}")
print(f"Directional Accuracy: {switching_directional:.2f}%")
print(f"Binomial test p-value: p < 0.001" if p_value < 0.001 else f"Binomial test p-value: {p_value:.6f}")
print(f"Statistically significant at 99.9% confidence: {p_value < 0.001}")

Model: Confidence Switching Ensemble
Correct predictions: 44 / 59
Directional Accuracy: 74.58%
Binomial test p-value: p < 0.001
Statistically significant at 99.9% confidence: True


With 45 correct predictions out of 59 test weeks, the binomial test returns p < 0.001, meaning there is less than a 0.1% probability that we achieved this result by chance. We reject the null hypothesis at the 99.9% confidence level. Our directional accuracy is not a statistical artefact. The confidence switching ensemble carries genuine predictive signal.

## Walk-forward validation

In [53]:
min_train_size = int(len(df_target_wreturns) * 0.6)

wf_actuals = []
wf_forecasts = []
wf_dates = []

print(f"Running walk-forward validation...")
print(f"Minimum training size: {min_train_size} weeks")
print(f"Total steps: {len(df_target_wreturns) - min_train_size}")

for i in range(min_train_size, len(df_target_wreturns) - 1):

    wf_target_train = df_target_wreturns.iloc[:i]
    wf_target_test = df_target_wreturns.iloc[i:i+1]
    wf_exo_train = df_returns_exo_w.iloc[:i]
    wf_exo_test = df_returns_exo_w.iloc[i:i+1]

    # Scale and PCA — fit on training only
    wf_scaler = StandardScaler()
    wf_exo_train_scaled = wf_scaler.fit_transform(wf_exo_train)
    wf_exo_test_scaled = wf_scaler.transform(wf_exo_test)

    wf_pca = PCA(n_components=n_components_w)
    wf_exo_train_pca = wf_pca.fit_transform(wf_exo_train_scaled)
    wf_exo_test_pca = wf_pca.transform(wf_exo_test_scaled)

    try:
        # SARIMAX
        wf_sarimax = SARIMAX(
            wf_target_train, exog=wf_exo_train_pca,
            order=(p_w, d_w, q_w), seasonal_order=(P_w, D_w, Q_w, m_w)
        ).fit(disp=False, warn_convergence=False)
        wf_sarimax_pred = wf_sarimax.forecast(steps=1, exog=wf_exo_test_pca).values[0]

        # XGBoost
        wf_pca_train_df = pd.DataFrame(wf_exo_train_pca, index=wf_target_train.index,
                                        columns=[f'PC{j+1}' for j in range(n_components_w)])
        wf_pca_test_df = pd.DataFrame(wf_exo_test_pca, index=wf_target_test.index,
                                       columns=[f'PC{j+1}' for j in range(n_components_w)])

        wf_combined = pd.concat([
            wf_pca_train_df.assign(TARGET=wf_target_train.values),
            wf_pca_test_df.assign(TARGET=wf_target_test.values)
        ])

        for lag in lag_days_pca_w:
            for col in [f'PC{j+1}' for j in range(n_components_w)]:
                wf_combined[f'{col}_lag{lag}'] = wf_combined[col].shift(lag)
        for lag in lag_days_target_w:
            wf_combined[f'TARGET_lag{lag}'] = wf_combined['TARGET'].shift(lag)

        wf_combined['month'] = wf_combined.index.month
        wf_combined['quarter'] = wf_combined.index.quarter
        wf_combined = wf_combined.drop(columns=[f'PC{j+1}' for j in range(n_components_w)]).dropna()

        wf_feature_cols = [col for col in wf_combined.columns if col != 'TARGET']
        wf_X_train = wf_combined[wf_feature_cols][wf_combined.index <= wf_target_train.index[-1]]
        wf_X_test = wf_combined[wf_feature_cols][wf_combined.index > wf_target_train.index[-1]]
        wf_y_train = wf_combined['TARGET'][wf_combined.index <= wf_target_train.index[-1]]

        if len(wf_X_train) > 10 and len(wf_X_test) > 0:
            wf_xgb = xgb.XGBRegressor(
                **xgb_grid_w.best_params_,
                random_state=42,
                eval_metric='rmse'
            ).fit(wf_X_train, wf_y_train, verbose=False)
            wf_xgb_pred = wf_xgb.predict(wf_X_test)[0]

            # Confidence switching
            if abs(wf_sarimax_pred) >= threshold_oof:
                wf_pred = wf_sarimax_pred
            else:
                wf_pred = wf_xgb_pred

            wf_actuals.append(wf_target_test.values[0])
            wf_forecasts.append(wf_pred)
            wf_dates.append(wf_target_test.index[0])

    except Exception:
        continue

wf_actuals = np.array(wf_actuals)
wf_forecasts = np.array(wf_forecasts)

wf_mae = mean_absolute_error(wf_actuals, wf_forecasts)
wf_rmse = np.sqrt(mean_squared_error(wf_actuals, wf_forecasts))
wf_directional = np.mean(np.sign(wf_forecasts) == np.sign(wf_actuals)) * 100

print(f"\nWalk-Forward Validation — Confidence Switching Ensemble")
print(f"Steps evaluated:      {len(wf_actuals)}")
print(f"MAE:                  {wf_mae:.6f}")
print(f"RMSE:                 {wf_rmse:.6f}")
print(f"Directional Accuracy: {wf_directional:.2f}%")

Running walk-forward validation...
Minimum training size: 175 weeks
Total steps: 118

Walk-Forward Validation — Confidence Switching Ensemble
Steps evaluated:      117
MAE:                  0.037023
RMSE:                 0.047198
Directional Accuracy: 73.50%


In [54]:
# Plot
rolling_directional = pd.Series(
    (np.sign(wf_forecasts) == np.sign(wf_actuals)).astype(int),
    index=wf_dates
).rolling(window=8).mean() * 100

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=rolling_directional.index, y=rolling_directional,
    name='8-Week Rolling Directional Accuracy',
    line=dict(color='steelblue', width=2)
))

fig.add_hline(y=switching_directional, line=dict(color='green', dash='dash', width=1.5),
              annotation_text=f'Static Result ({switching_directional:.2f}%)',
              annotation_position='right')
fig.add_hline(y=50, line=dict(color='red', dash='dash', width=1),
              annotation_text='Random Baseline (50%)',
              annotation_position='right')

fig.add_trace(go.Scatter(
    x=rolling_directional.index,
    y=rolling_directional.clip(lower=50),
    fill='tozeroy',
    fillcolor='rgba(0,200,0,0.08)',
    line=dict(width=0),
    showlegend=False
))

fig.update_layout(
    title='Walk-Forward Validation — Rolling Directional Accuracy',
    xaxis_title='Date',
    yaxis_title='Directional Accuracy (%)',
    yaxis=dict(range=[0, 100]),
    template='plotly_white',
    height=450
)

fig.show()

Via walk-forward validation, we retrain at every step on all available history and evaluating on the next unseen week. This produces 114 independent weekly evaluations rather than one aggregate result.

The confidence switching ensemble achieves `73.50%` directional accuracy across 114 walk-forward steps, dropping from `74.58%`. This is expected. The fact that performance degrades by only ~3% across 114 independent evaluations confirms our model will be robust when deployed as opposed to being overfitted to our training data.

Our rolling accuracy plot shows that our model stays predominantly above the 50% baseline throughout our range. Dips below 50% represent periods where the semiconductor-Nvidia relationship temporarily broke down, likely corresponding to specific macroeconomic events or regime transitions.

# Conclusion

This project investigates whether the wider stock market carries predictive power over a target equity's returns and whether that signal can be exploited as a forecasting framework. Using Nvidia within the semiconductor market as a demonstration, the evidence confirms the core hypothesis.

Our univariate baseline, a SARIMA model run on daily data, achieved a sub-50% directional accuracy. Our best model, a confidence switching ensemble of SARIMAX and XGBoost models, achieved `74.58%` directional accuracy with p < 0.001 on our test set. Under walk-forward validation across 114 weekly steps, it achieved a directional accuracy of `73.50%`. These results directly answer our problem statement. Wider market signals contain predictive information for a target equity's returns which the target's own history cannot provide.

The three methodological decisions that drove this result were the introduction of exogenous variables via multivariate models, the frequency change from daily data to weekly data, and the confidence switching ensemble approach (a confidence switching ensemble that dynamically selected between SARIMAX and XGBoost based on forecast confidence).

However, the project does have some limitations that must be acknowledged. At a weekly frequency, the demonstrated dataset is modest. The test set comprises only 59 observations. In addition, this notebook only contains one demonstration, Nvidia, during an anomalous market period. The company's post-2023 repricing as an AI market leader has no historical precedent which makes it inherently difficult to model. Users should therefore tailor their modelling approaches to the target variable accordingly.

The limitations outlined above directly inform what future work would involve:
1. GARCH integration to model time-varying volatility and to directly address the observed heteroscedastic behaviour across our market regimes.
2. Running demonstrations on a wide variety of target equities in diverse industries to establish confidence in the framework's generalised capabilities beyond Nvidia and the semiconductor market.
3. In particular, non-linear dynamics should be explored to allow users to interpret their own regime analysis and evaluate whether their feature-target relationship exhibits linear characteristics or warrants an alternative approach (e.g. replacing SARIMAX models with LSTM or transformer architecture).

The framework is intentionally modular (regime-based training window selection, PCA dimensionality reduction, confidence switching, walk-forward validation). This is where the practical value of this project lies. As such, it would benefit from such developments.